# AI Language Translation Tool

**Project:** Language Translation  
**Author:** Saabeer  

---

### 📌 Project Overview
This notebook demonstrates the step-by-step development and testing of a modern **AI Language Translation Tool** featuring multi-language translation, speech synthesis (TTS), live word/character counters, and an iOS-inspired user interface.

#### 🎯 Objectives:
1. Allow users to enter text to translate.
2. Select a source language (or use Auto-Detect).
3. Select a target language from 15+ supported languages.
4. Process translation requests using `deep-translator` with error handling.
5. Provide character and word counting, swap languages, copy to clipboard, and Text-to-Speech (TTS).
6. Build an interactive iOS-styled web UI using **Gradio** that runs seamlessly on Google Colab and local machines.

## 1. Environment Setup & Runtime Inspection
Let us first inspect our Python runtime environment, OS platform, and version to ensure compatibility.

In [1]:
import sys
import platform

print("Python Version:", sys.version)
print("Operating System Platform:", platform.platform())

Python Version: 3.13.15
Operating System Platform: Linux-6.6.122+-x86_64-with-glibc2.39


## 2. Package Installation
We install the core required libraries:
- `gradio`: For creating the interactive user interface.
- `deep-translator`: Python library to translate text using translation services (e.g. GoogleTranslator, MyMemoryTranslator).
- `gTTS`: Google Text-to-Speech library to convert translated text to audio.

In [2]:
!pip install --quiet gradio deep-translator gTTS

## 3. Import Libraries
We import all the required modules and verify their availability.

In [3]:
from typing import Tuple, Dict
import gradio as gr
from deep_translator import GoogleTranslator, MyMemoryTranslator
from deep_translator.exceptions import (
    TooManyRequests,
    LanguageNotSupportedException,
    TranslationNotFound,
    RequestError
)
from gtts import gTTS
import tempfile
import os

print("✅ Gradio version:", gr.__version__)
print("✅ All translation and speech libraries imported successfully.")

✅ Gradio version: 6.27.0
✅ All translation and speech libraries imported successfully.


## 4. Language Configuration
We define our supported languages, including major Indian languages (Telugu, Hindi, Tamil, Kannada, Malayalam, Marathi, Bengali, Gujarati) and international languages (English, Spanish, French, German, Japanese, Chinese, Arabic).

Each language is mapped to its standard ISO 639-1 code.

In [4]:
# Supported target languages with their ISO 639-1 language codes
SUPPORTED_LANGUAGES: Dict[str, str] = {
    "English": "en",
    "Telugu": "te",
    "Hindi": "hi",
    "Tamil": "ta",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Marathi": "mr",
    "Bengali": "bn",
    "Gujarati": "gu",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Japanese": "ja",
    "Chinese": "zh-CN",
    "Arabic": "ar"
}

# Source languages include 'Auto Detect'
SOURCE_LANGUAGES: Dict[str, str] = {
    "Auto Detect": "auto",
    **SUPPORTED_LANGUAGES
}

# Target languages must be an explicit language
TARGET_LANGUAGES: Dict[str, str] = SUPPORTED_LANGUAGES

# Secondary mapping for MyMemoryTranslator fallback (handles IP rate limits)
_MYMEMORY_CODE_MAP: Dict[str, str] = {
    "auto": "autodetect",
    "en": "en-GB",
    "te": "te-IN",
    "hi": "hi-IN",
    "ta": "ta-IN",
    "kn": "kn-IN",
    "ml": "ml-IN",
    "mr": "mr-IN",
    "bn": "bn-IN",
    "gu": "gu-IN",
    "es": "es-ES",
    "fr": "fr-FR",
    "de": "de-DE",
    "ja": "ja-JP",
    "zh-CN": "zh-CN",
    "ar": "ar-SA"
}

MAX_CHAR_LIMIT = 5000

print(f"Configured {len(SOURCE_LANGUAGES)} source options and {len(TARGET_LANGUAGES)} target languages.")

Configured 16 source options and 15 target languages.


## 5. Helper & Translation Functions
We implement modular functions:
1. `get_language_code`: Safely resolves language names to ISO codes.
2. `count_text_stats`: Computes character and word counts.
3. `translate_text`: Validates input, executes translation via `GoogleTranslator`, and includes automatic fallback to `MyMemoryTranslator` if Google rate-limits (HTTP 429).

In [5]:
def get_language_code(language_name_or_code: str, is_source: bool = False) -> str:
    """
    Converts a language name (e.g. 'Telugu') or code ('te') into a valid ISO code.
    """
    if not language_name_or_code:
        raise ValueError("Language selection cannot be empty.")

    clean_name = language_name_or_code.strip()
    valid_codes = set(SOURCE_LANGUAGES.values()) if is_source else set(TARGET_LANGUAGES.values())
    
    if clean_name.lower() in [c.lower() for c in valid_codes]:
        return clean_name

    lang_dict = SOURCE_LANGUAGES if is_source else TARGET_LANGUAGES
    for name, code in lang_dict.items():
        if name.lower() == clean_name.lower():
            return code

    raise ValueError(f"Unsupported language: '{language_name_or_code}'.")


def count_text_stats(text: str) -> Tuple[int, int]:
    """
    Returns (character_count, word_count) for the provided text.
    """
    if not text:
        return 0, 0
    return len(text), len(text.split())


def translate_text(
    text: str,
    source_language: str = "Auto Detect",
    target_language: str = "Telugu"
) -> str:
    """
    Translates text with full validation and exception handling.
    """
    # 1. Empty or whitespace check
    if text is None or not text.strip():
        return ""

    clean_text = text.strip()

    # 2. Maximum character check
    if len(clean_text) > MAX_CHAR_LIMIT:
        return f"Error: Text exceeds maximum limit of {MAX_CHAR_LIMIT} characters."

    # 3. Resolve language codes
    try:
        src_code = get_language_code(source_language, is_source=True)
        tgt_code = get_language_code(target_language, is_source=False)
    except ValueError as err:
        return f"Language Error: {str(err)}"

    # 4. Identity translation check
    if src_code != "auto" and src_code.lower() == tgt_code.lower():
        return clean_text

    # 5. Translation execution with fallback
    try:
        translator = GoogleTranslator(source=src_code, target=tgt_code)
        result = translator.translate(clean_text)
        return result if result is not None else ""
    except TooManyRequests:
        # Fallback to MyMemoryTranslator if rate limited
        try:
            mem_src = _MYMEMORY_CODE_MAP.get(src_code, src_code)
            mem_tgt = _MYMEMORY_CODE_MAP.get(tgt_code, tgt_code)
            fallback = MyMemoryTranslator(source=mem_src, target=mem_tgt)
            res = fallback.translate(clean_text)
            return res if res is not None else ""
        except Exception as fb_err:
            return f"Translation Service Error: {str(fb_err)}"
    except (LanguageNotSupportedException, TranslationNotFound) as e:
        return f"Translation Error: {str(e)}"
    except RequestError as e:
        return f"Network Error: {str(e)}"
    except Exception as e:
        return f"Unexpected Error: {str(e)}"

print("✅ Translation functions loaded successfully.")

✅ Translation functions loaded successfully.


## 6. Translation Testing
Let us test the required translation test cases specified in the internship guidelines:
1. **English → Telugu**: `"Hello, how are you?"`
2. **Telugu → English**: `"మీరు ఎలా ఉన్నారు?"`
3. **English → Hindi**: `"Good morning"`
4. **Hindi → English**: `"आप कैसे हैं?"`
5. Edge cases: Empty input, same source & target, stats.

In [6]:
# Test 1: English -> Telugu
text_en = "Hello, how are you?"
res_te = translate_text(text_en, source_language="English", target_language="Telugu")
print("Test 1: English -> Telugu")
print("Input :", text_en)
print("Output:", res_te)
print("Stats :", count_text_stats(text_en))
print("-" * 50)

Test 1: English -> Telugu
Input : Hello, how are you?
Output: హలో, ఎలా ఉన్నారు?
Stats : (19, 4)
--------------------------------------------------


In [7]:
# Test 2: Telugu -> English
text_te = "మీరు ఎలా ఉన్నారు?"
res_en = translate_text(text_te, source_language="Telugu", target_language="English")
print("Test 2: Telugu -> English")
print("Input :", text_te)
print("Output:", res_en)
print("-" * 50)

Test 2: Telugu -> English
Input : మీరు ఎలా ఉన్నారు?
Output: How are you?
--------------------------------------------------


In [8]:
# Test 3: English -> Hindi
text_gm = "Good morning"
res_hi = translate_text(text_gm, source_language="English", target_language="Hindi")
print("Test 3: English -> Hindi")
print("Input :", text_gm)
print("Output:", res_hi)
print("-" * 50)

Test 3: English -> Hindi
Input : Good morning
Output: शुभ प्रभात
--------------------------------------------------


In [9]:
# Test 4: Hindi -> English
text_hi = "आप कैसे हैं?"
res_en2 = translate_text(text_hi, source_language="Hindi", target_language="English")
print("Test 4: Hindi -> English")
print("Input :", text_hi)
print("Output:", res_en2)
print("-" * 50)

Test 4: Hindi -> English
Input : आप कैसे हैं?
Output: How are you?
--------------------------------------------------


In [10]:
# Test 5: Edge Cases & Validation
print("Test 5: Edge Cases")
print("Empty input:", repr(translate_text("")))
print("Whitespace only:", repr(translate_text("    ")))
print("Same source and target:", translate_text("Artificial Intelligence", "English", "English"))
print("Invalid language:", translate_text("Test", "Kryptonian", "English"))
print("Word & char count for 'Artificial Intelligence':", count_text_stats("Artificial Intelligence"))

Test 5: Edge Cases
Empty input: ''
Whitespace only: ''
Same source and target: Artificial Intelligence
Invalid language: Language Error: Unsupported language: 'Kryptonian'.
Word & char count for 'Artificial Intelligence': (23, 2)


## 7. Text-to-Speech (TTS) Integration
We implement `generate_tts_audio` using `gTTS` (Google Text-to-Speech) to generate audio pronunciation for translated text across all 15 supported languages.

In [11]:
def generate_tts_audio(text: str, target_language: str = "Telugu"):
    """
    Synthesizes speech from translated text using Google Text-to-Speech (gTTS).
    """
    if not text or not text.strip():
        return None

    clean_text = text.strip()
    if clean_text.startswith("Error:") or clean_text.startswith("Language Error:"):
        return None

    try:
        lang_code = get_language_code(target_language, is_source=False)
        tts = gTTS(text=clean_text, lang=lang_code, slow=False)
        temp_file = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        temp_path = temp_file.name
        temp_file.close()
        tts.save(temp_path)
        return temp_path
    except Exception as e:
        print(f"[TTS Warning]: {e}")
        return None

# Quick TTS test
test_audio = generate_tts_audio("నమస్కారం", "Telugu")
if test_audio and os.path.exists(test_audio):
    print("✅ TTS audio generated and verified successfully.")

✅ TTS audio generated and verified successfully.


## 8. Interactive iOS-Style Gradio Interface
We construct the web interface designed with Apple Human Interface Guidelines:
- **Sleek Layout**: Clean iOS-style navigation header, cards, and San Francisco typography.
- **Controls**: Translate (􀊄), Swap (⇄), Clear (✕), Listen (🔊).
- **Live Metrics**: Real-time character & word counter chips.
- **Translation History**: Inset grouped history list.

In [12]:
from datetime import datetime
import inspect

IOS_CSS = """
:root {
    --ios-bg: #F2F2F7; --ios-card-bg: #FFFFFF; --ios-blue: #007AFF;
    --ios-gray-fill: #E5E5EA; --ios-gray-text: #8E8E93; --ios-dark-text: #1C1C1E;
    --ios-border: rgba(60, 60, 67, 0.12); --ios-pill-radius: 9999px;
}
body { background-color: var(--ios-bg) !important; font-family: -apple-system, BlinkMacSystemFont, sans-serif !important; }
.gradio-container { max-width: 960px !important; margin: 20px auto !important; }
.ios-header { text-align: center; padding: 26px 20px; background: #FFFFFF; border-radius: 24px; box-shadow: 0 8px 30px rgba(0,0,0,0.04); border: 1px solid var(--ios-border); margin-bottom: 20px; }
.ios-pill-badge { display: inline-flex; background: rgba(0,122,255,0.1); color: #007AFF; padding: 4px 14px; border-radius: 9999px; font-size: 0.8rem; font-weight: 700; margin-bottom: 8px; }
.ios-header-title { font-size: 2.3rem !important; font-weight: 800 !important; color: #1C1C1E !important; margin: 0 !important; }
.ios-header-subtitle { color: #8E8E93 !important; font-size: 1rem !important; margin: 4px 0 0 0 !important; }
button[variant="primary"] { background: #007AFF !important; border-radius: 9999px !important; font-weight: 700 !important; font-size: 1rem !important; }
button[variant="secondary"] { background: #E5E5EA !important; color: #007AFF !important; border-radius: 9999px !important; font-weight: 600 !important; }
"""

def on_translate(text, source_lang, target_lang, history):
    if not text or not text.strip():
        return "", "📊 0 characters • 0 words", None, history, history
    translated = translate_text(text, source_lang, target_lang)
    chars, words = count_text_stats(text)
    stats = f"📊 {chars} characters • {words} words"
    audio_path = None
    if translated and not translated.startswith("Error:") and not translated.startswith("Language Error:"):
        audio_path = generate_tts_audio(translated, target_lang)
    timestamp = datetime.now().strftime("%H:%M:%S")
    new_entry = [timestamp, source_lang, target_lang, text.strip(), translated]
    updated_history = [new_entry] + (history or [])
    return translated, stats, audio_path, updated_history, updated_history

def on_swap(source_lang, target_lang, input_text, output_text):
    new_source = target_lang
    new_target = "English" if source_lang == "Auto Detect" else source_lang
    new_input = output_text if output_text and not output_text.startswith("Error:") else input_text
    chars, words = count_text_stats(new_input)
    return new_source, new_target, new_input, "", f"📊 {chars} characters • {words} words", None

def on_clear():
    return "", "", "📊 0 characters • 0 words", None

def on_listen(output_text, target_lang):
    return generate_tts_audio(output_text, target_lang)

def on_input_change(text):
    chars, words = count_text_stats(text)
    return f"📊 {chars} characters • {words} words"

# Dynamic copy button kwarg across Gradio versions
tb_sig = inspect.signature(gr.Textbox.__init__).parameters
copy_kwarg = {'buttons': ['copy']} if 'buttons' in tb_sig else ({'show_copy_button': True} if 'show_copy_button' in tb_sig else {})

with gr.Blocks(title="Translate — AI Language Translation") as demo:
    history_state = gr.State([])
    gr.HTML("""
    <div class="ios-header">
        <div class="ios-pill-badge">✦ Neural Translate</div>
        <h1 class="ios-header-title">Translate</h1>
        <p class="ios-header-subtitle">Instant speech and text translation across 15+ languages</p>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=1):
            source_dd = gr.Dropdown(choices=list(SOURCE_LANGUAGES.keys()), value="English", label="From")
            inp_box = gr.Textbox(label="Text to Translate", placeholder="Enter text here...", lines=6, **copy_kwarg)
            stats_lbl = gr.Markdown("📊 0 characters • 0 words")
        with gr.Column(scale=1):
            target_dd = gr.Dropdown(choices=list(TARGET_LANGUAGES.keys()), value="Telugu", label="To")
            out_box = gr.Textbox(label="Translation", placeholder="Translation appears here...", lines=6, interactive=False, **copy_kwarg)
            audio_box = gr.Audio(label="🔊 Speech Pronunciation", type="filepath", interactive=False)
    with gr.Row():
        trans_btn = gr.Button("􀊄 Translate", variant="primary", scale=2)
        swap_btn = gr.Button("⇄ Swap", variant="secondary", scale=1)
        clear_btn = gr.Button("✕ Clear", variant="secondary", scale=1)
        listen_btn = gr.Button("🔊 Listen", variant="secondary", scale=1)
    with gr.Accordion("📜 Translation History", open=True):
        hist_table = gr.Dataframe(headers=["Time", "From", "To", "Original Text", "Translated Text"], datatype=["str"]*5, value=[], interactive=False, wrap=True)
        clear_hist_btn = gr.Button("🗑️ Clear History", size="sm", variant="stop")
    
    # Wire events
    inp_box.change(fn=on_input_change, inputs=[inp_box], outputs=[stats_lbl])
    trans_btn.click(fn=on_translate, inputs=[inp_box, source_dd, target_dd, history_state], outputs=[out_box, stats_lbl, audio_box, hist_table, history_state])
    inp_box.submit(fn=on_translate, inputs=[inp_box, source_dd, target_dd, history_state], outputs=[out_box, stats_lbl, audio_box, hist_table, history_state])
    swap_btn.click(fn=on_swap, inputs=[source_dd, target_dd, inp_box, out_box], outputs=[source_dd, target_dd, inp_box, out_box, stats_lbl, audio_box])
    clear_btn.click(fn=on_clear, inputs=[], outputs=[inp_box, out_box, stats_lbl, audio_box])
    listen_btn.click(fn=on_listen, inputs=[out_box, target_dd], outputs=[audio_box])
    clear_hist_btn.click(fn=lambda: ([], []), inputs=[], outputs=[hist_table, history_state])

print("✅ iOS-style Gradio application constructed and ready to launch.")

✅ Gradio application constructed and ready to launch.


## 9. Launch Application
Run the cell below to launch the web application. In Google Colab, `share=True` creates a temporary public URL for easy browser testing and video demonstration.

In [ ]:
# In Google Colab, share=True creates a public Gradio link
# Set share=False when running locally
demo.launch(share=True)